# 05 — Charleston ice-plant fire event study (T4.2)

Re-runs `portgaps.eventstudy.run_charleston` and shows the main tables and figures. Design, donors, inference, robustness and limits: `docs/methods/event_study_charleston.md`.

Event dates come from the closure sequence (closed 2019-12-20 by fire; reopened 2020-09-12). During = 2019-12..2020-09 (December has ~11 closed days), after = 2020-10..2024-12, pre = 2016-01..2019-11. Charleston = ODFW harbor-level table, all species, nominal dollars; it stops at 2024-12 because Charleston is group-level from 2025. Main donor pool = 5 Oregon ODFW units; northern CA harbors are a robustness check only.

In [1]:
import sys, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 70); pd.set_option('display.max_rows', 200)
from portgaps.eventstudy import run_charleston as run
E, R, P = run.main()
print(len(E), 'estimate rows;', len(R), 'robustness rows;', len(P), 'placebo rows')

45 estimate rows; 99 robustness rows; 208 placebo rows


## Donor pool
`used` = in the pool; reasons for every exclusion. Distances are charted-equivalent nm (sea_nm × 1.06) from Charleston.

In [2]:
D = pd.read_csv('../outputs/tables/t42_donors.csv')
D[['pool','scheme','unit_id','inventory_harbors_nm','used','excluded_reason','notes']]

                 pool  ...                                          notes
0             main_OR  ...                                            NaN
1             main_OR  ...                                            NaN
2             main_OR  ...                                            NaN
3             main_OR  ...                                            NaN
4             main_OR  ...                                            NaN
5             main_OR  ...                                            NaN
6             main_OR  ...                                            NaN
7             main_OR  ...                                            NaN
8   check_northern_CA  ...                                            NaN
9   check_northern_CA  ...                                            NaN
10  check_northern_CA  ...  partial_confidential months (lower bound): 76
11  check_northern_CA  ...                                            NaN
12  check_northern_CA  ...            

## Main estimates
E1/E2 Charleston value (SC/DiD); E3 pounds; E4 Dungeness crab; E5 shift test (Waldport–Winchester Bay unit); E6 Charleston + that unit combined. `p_time` uses pre-period fake events only (main); `p_time_incl_post` adds fake Dec events 2021–2023 (assumes no lasting effect).

In [3]:
cols = ['estimate_id','method','window','observed','counterfactual','pct_change','p_space_ratio','p_space_pct','p_time','p_time_incl_post']
E[cols].round(3)

   estimate_id             method  ... p_time  p_time_incl_post
0           E1  synthetic_control  ...  0.333             0.167
1           E1  synthetic_control  ...    NaN               NaN
2           E1  synthetic_control  ...    NaN               NaN
3           E1  synthetic_control  ...    NaN               NaN
4           E2         window_DiD  ...  0.250             0.286
5           E2         window_DiD  ...  0.250               NaN
6           E2         window_DiD  ...  0.250               NaN
7           E2         window_DiD  ...  0.250               NaN
8           E2         window_DiD  ...  0.250               NaN
9           E2         window_DiD  ...    NaN               NaN
10          E3  synthetic_control  ...  0.333             0.500
11          E3  synthetic_control  ...    NaN               NaN
12          E3  synthetic_control  ...    NaN               NaN
13          E3  synthetic_control  ...    NaN               NaN
14          E3         window_DiD  ...  

## Robustness (R1–R11) and power (R10)

In [4]:
R[['robustness_id','check','label','window','pct_change','p_time']].round(3)

   robustness_id                                     check  ... pct_change p_time
0             R1                             placebo dates  ...     -0.339  0.333
1             R1                             placebo dates  ...     -0.207  0.333
2             R1                             placebo dates  ...     -0.357  0.333
3             R1                             placebo dates  ...      0.037  1.000
4             R1                             placebo dates  ...     -0.295  0.333
5             R1                             placebo dates  ...     -0.345  0.250
6             R1                             placebo dates  ...     -0.408  0.250
7             R1                             placebo dates  ...     -0.008  1.000
8             R1                             placebo dates  ...     -0.302  0.250
9             R2                       leave-one-out donor  ...     -0.339    NaN
10            R2                       leave-one-out donor  ...     -0.328  0.250
11            R2

In [5]:
c = [x for x in R.columns if x.startswith('p_if') or x in ('label','n_placebos','placebo_log_range','smallest_loss_outside_all_placebos','min_attainable_p')]
R[R.robustness_id == 'R10'][c].round(3)

                                          label  ...  p_if_true_loss_90
92           E1 SC, placebo-in-space (5 donors)  ...              0.167
93  E1 SC, placebo-in-time pre (Dec 2017, 2018)  ...              0.333
94            E1 SC, placebo-in-time pre + post  ...              0.167
95          E2 DiD, pre windows (Dec 2016-2018)  ...              0.250
96                   E2 DiD, pre + post windows  ...              0.143
97  E5 SC Winchester Bay unit, placebo-in-space  ...              0.167
98             E6 SC combined, placebo-in-space  ...              0.167

[7 rows x 9 columns]

## Figures
![treated vs counterfactual](../outputs/figures/t42_treated_vs_counterfactual.png)

![Winchester Bay shift](../outputs/figures/t42_winchester_shift.png)

![placebos](../outputs/figures/t42_placebo_distribution.png)

## Plain-language result
Charleston landed about one-third less than comparable Oregon ports predict during the closure (SC −34%, DiD −35%; crab −36% to −41%; pounds −21% to −30%). The Winchester Bay unit shows no gain (+4% SC, −1% DiD), and Charleston + that unit still fall about 30%, so the landings did not shift to the documented backup. But Charleston stayed about a third below the same comparison in every year after the plant reopened, and with five donors no test can come out below p = 0.17. Low power; not attributable to the ice closure alone; never a significant effect.